# Vendor C: Custom Platform (Asian Market)

In [0]:
# IMPORT STATEMENT
from pyspark.sql.functions import (
    input_file_name, current_timestamp, lit, col, when, coalesce, from_json, regexp_replace
)

## Preparing Bronze Schema

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS fixing_the_broken_data_pipeline;
USE CATALOG fixing_the_broken_data_pipeline;
CREATE SCHEMA IF NOT EXISTS bronze;

In [0]:
# COMMON VARIABLES
bucket = "vendor-data-24042026"
vendor_data_dir = "vendor_c_customplatform_data"
target_table = "bronze.orders_vendor_c_customplatform"

records = "orders_*.tsv"
path = f"s3a://{bucket}/{vendor_data_dir}/{records}"

In [0]:
# READING THE DATA
df_raw = (
    spark.read
    .options(header=True, delimiter='\t', inferSchema=True)
    .csv(f"{path}")
)

df_raw.toPandas().head(5)

,order_number,created_at,customer_info,line_items,payment_details,device_type
0,C916233,2025-03-28 11:10:09,"""{""""email"""":""""tanaka1460@example.jp"""",""""countr...","""[{""""sku"""":""""WIDGET-222"""",""""qty"""":3,""""price"""":...","""{""""method"""":""""mobile_wallet"""",""""currency"""":""""...",desktop
1,C912516,2025-03-28 06:30:35,"""{""""email"""":""""park417@example.kr"""",""""country""""...","""[{""""sku"""":""""WIDGET-112"""",""""qty"""":2,""""price"""":...","""{""""method"""":""""mobile_wallet"""",""""currency"""":""""...",mobile
2,C917823,2025-03-28 17:50:58,"""{""""email"""":""""kim1010@example.jp"""",""""country""""...","""[{""""sku"""":""""WIDGET-380"""",""""qty"""":5,""""price"""":...","""{""""method"""":""""credit_card"""",""""currency"""":""""JP...",desktop
3,C909214,2025-03-28 18:15:09,"""{""""email"""":""""lee7691@example.kr"""",""""country""""...","""[{""""sku"""":""""WIDGET-269"""",""""qty"""":3,""""price"""":...","""{""""method"""":""""mobile_wallet"""",""""currency"""":""""...",tablet
4,C916674,2025-03-28 05:15:00,"""{""""email"""":""""kim810@example.kr"""",""""country"""":...","""[{""""sku"""":""""WIDGET-370"""",""""qty"""":3,""""price"""":...","""{""""method"""":""""bank_transfer"""",""""currency"""":""""...",mobile


In [0]:
# REMOVING THE QUOTES
df_raw = (
    spark.read
    .options(header=True, delimiter='\t', inferSchema=True, quote = '"', escape = '"')
    .csv(f"{path}")
)

df_raw.toPandas().head(5)

,order_number,created_at,customer_info,line_items,payment_details,device_type
0,C916233,2025-03-28 11:10:09,"{""email"":""tanaka1460@example.jp"",""country"":""JP...","[{""sku"":""WIDGET-222"",""qty"":3,""price"":25879}]","{""method"":""mobile_wallet"",""currency"":""JPY"",""am...",desktop
1,C912516,2025-03-28 06:30:35,"{""email"":""park417@example.kr"",""country"":""KR"",""...","[{""sku"":""WIDGET-112"",""qty"":2,""price"":248173}]","{""method"":""mobile_wallet"",""currency"":""KRW"",""am...",mobile
2,C917823,2025-03-28 17:50:58,"{""email"":""kim1010@example.jp"",""country"":""JP"",""...","[{""sku"":""WIDGET-380"",""qty"":5,""price"":11907}]","{""method"":""credit_card"",""currency"":""JPY"",""amou...",desktop
3,C909214,2025-03-28 18:15:09,"{""email"":""lee7691@example.kr"",""country"":""KR"",""...","[{""sku"":""WIDGET-269"",""qty"":3,""price"":38113}]","{""method"":""mobile_wallet"",""currency"":""KRW"",""am...",tablet
4,C916674,2025-03-28 05:15:00,"{""email"":""kim810@example.kr"",""country"":""KR"",""p...","[{""sku"":""WIDGET-370"",""qty"":3,""price"":52212}]","{""method"":""bank_transfer"",""currency"":""KRW"",""am...",mobile


In [0]:
df_bronze = (
    df_raw
        .withColumn("_source_file", col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", current_timestamp())
        .withColumn("_source_vendor", lit("Vendor A"))
)

# SAVING INTO DELTA FORMAT
df_bronze.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(target_table)

# CHECKING
df_bronze.toPandas().head(5)

,order_number,created_at,customer_info,line_items,payment_details,device_type,_source_file,_ingestion_timestamp,_source_vendor
0,C916233,2025-03-28 11:10:09,"{""email"":""tanaka1460@example.jp"",""country"":""JP...","[{""sku"":""WIDGET-222"",""qty"":3,""price"":25879}]","{""method"":""mobile_wallet"",""currency"":""JPY"",""am...",desktop,s3a://vendor-data-24042026/vendor_c_customplat...,2026-05-01 01:50:23.965661,Vendor A
1,C912516,2025-03-28 06:30:35,"{""email"":""park417@example.kr"",""country"":""KR"",""...","[{""sku"":""WIDGET-112"",""qty"":2,""price"":248173}]","{""method"":""mobile_wallet"",""currency"":""KRW"",""am...",mobile,s3a://vendor-data-24042026/vendor_c_customplat...,2026-05-01 01:50:23.965661,Vendor A
2,C917823,2025-03-28 17:50:58,"{""email"":""kim1010@example.jp"",""country"":""JP"",""...","[{""sku"":""WIDGET-380"",""qty"":5,""price"":11907}]","{""method"":""credit_card"",""currency"":""JPY"",""amou...",desktop,s3a://vendor-data-24042026/vendor_c_customplat...,2026-05-01 01:50:23.965661,Vendor A
3,C909214,2025-03-28 18:15:09,"{""email"":""lee7691@example.kr"",""country"":""KR"",""...","[{""sku"":""WIDGET-269"",""qty"":3,""price"":38113}]","{""method"":""mobile_wallet"",""currency"":""KRW"",""am...",tablet,s3a://vendor-data-24042026/vendor_c_customplat...,2026-05-01 01:50:23.965661,Vendor A
4,C916674,2025-03-28 05:15:00,"{""email"":""kim810@example.kr"",""country"":""KR"",""p...","[{""sku"":""WIDGET-370"",""qty"":3,""price"":52212}]","{""method"":""bank_transfer"",""currency"":""KRW"",""am...",mobile,s3a://vendor-data-24042026/vendor_c_customplat...,2026-05-01 01:50:23.965661,Vendor A


In [0]:
%sql
DESC bronze.orders_vendor_b_magento_data

col_name,data_type,comment
order_id,string,null
order_date,date,null
customer_email,string,null
product_code,string,null
quantity,int,null
unit_price,string,null
total_amount,string,null
payment_method,string,null
country_code,string,null
vat_rate,string,null
